# Phase 5-C — TA-01 fill-feasibility diagnostic (Development only)

**Not a backtest.** Counts only: how often TA-01's order intents find no tradable 1m bar within 2 minutes, and how often the participation cap blocks 100 shares. No prices, PnL, returns or strategy conclusions. Holdout: NO_ACCESS. No PREPARE / PREFLIGHT / CLAIM / EXECUTE of the research runner, no Batch.

Retired and refused in every mode: `RA-IBKR_TSM-TA01_FILLDIAG-0001` (PRE_DATA_INPUT_PATH_FAILURE) and `RA-IBKR_TSM-TA01_FILLDIAG-0002` (USED, SUCCEEDED). Also retired: `RA-IBKR_TSM-TA01Q8_FILLDIAG-0001` (USED, SUCCEEDED). Current draft: `RA-IBKR_TSM-TA01Q8X_FILLDIAG-0001` for **P5C-TA01-Q8X** (8 units; exits persist to the session end; POST_DIAGNOSTIC_VARIANT, EXPLORATORY_ONLY). Core's full-minute participation check stays an open **BLOCKER**.

1. **PREPARE** (default): does not mount Google Drive, reads no data and no credential; prints only the authorization id, document / scope SHA-256 and the exact approval line.
2. hh4832 personally posts that line as a comment on the PR (not via any App or agent).
3. Fresh Runtime, `MODE="RUN"`, bare `APPROVAL_URL` (no Markdown link), typed id. RUN refuses paths other than those pinned in the authorization, checks the approval, scope, input paths and metadata **before** creating anything on Drive, then writes a durable attempt marker before any Development data can be read. Success → `MANIFEST.json`; failure → `FAILURE_MANIFEST.json`; both write-once. One attempt per authorization; a failed attempt needs a new authorization.

In [ ]:
# DO NOT set MODE="RUN" before hh4832 has posted the approval comment for AUTHORIZATION_ID and it is reviewed.
MODE = "PREPARE"            # PREPARE (no Drive, no data, no credential) | RUN
AUTHORIZATION_ID = "RA-IBKR_TSM-TA01Q8X_FILLDIAG-0001"
CODE_SHA = ""               # reviewed main commit that contains this authorization draft (40 hex)
CORE_SHA = "50ea3bfa42df7708dd3a1b21f17798c206be054e"
APPROVAL_EXPIRES_UTC = ""   # PREPARE: e.g. "2026-11-26T00:00:00Z" (<= 336 h after posting)
APPROVAL_URL = ""           # RUN: bare URL only, e.g. https://github.com/hh4832/strategy-research-agent/pull/<n>#issuecomment-<id>
RECOVERY_ARCHIVE = "/content/drive/MyDrive/00Quant_Research/data/development_recovery_archives/TSM_DEV_1M_20161019_20241231_ca9f87df83f31bd24211_20261110T142034_335006Z"
ORIGINAL_PREPARATION_ARCHIVE = "/content/drive/MyDrive/Quant_Research/strategy-research-agent/phase5c1/20261111_230118_690789"
CONFIRM_AUTHORIZATION = ""  # RUN: type the full AUTHORIZATION_ID
DRIVE_ROOT = "/content/drive/MyDrive/Quant_Research/strategy-research-agent/phase5c_ta01_fill_diagnostic"  # fixed


In [ ]:
import os, sys, re, json, pathlib, subprocess
if MODE not in ("PREPARE", "RUN"):
    raise RuntimeError("Unsupported mode")
RETIRED = ("RA-IBKR_TSM-TA01_FILLDIAG-0001", "RA-IBKR_TSM-TA01_FILLDIAG-0002", "RA-IBKR_TSM-TA01Q8_FILLDIAG-0001")
if AUTHORIZATION_ID in RETIRED:
    raise RuntimeError("retired authorization (pre-data failure or already used); never re-run")
if not re.fullmatch(r"[0-9a-f]{40}", CODE_SHA):
    raise RuntimeError("A reviewed code SHA is required")
if MODE == "RUN":
    if CONFIRM_AUTHORIZATION != AUTHORIZATION_ID:
        raise RuntimeError("Type the full authorization id to RUN")
    if not re.fullmatch(r"https://github\.com/hh4832/strategy-research-agent/(?:pull|issues)/[1-9][0-9]*#issuecomment-[1-9][0-9]*", APPROVAL_URL):
        raise RuntimeError("APPROVAL_URL must be the bare comment URL (no Markdown link, brackets or spaces)")
SECRETS = []
def clean(text):
    for s in SECRETS:
        text = text.replace(s, "[REDACTED]")
    text = re.sub(r"(gh[pousr]_[A-Za-z0-9]{20,}|github_pat_[A-Za-z0-9_]{20,})", "[REDACTED]", text)
    return re.sub(r"(?i)\b(authorization|token|password|secret)(\s*[:=]\s*)(?!RA-[A-Z0-9_]+(?:-[A-Z0-9_]+)*-[0-9]{4}\b)\S+", r"\1\2[REDACTED]", text)
def run(cmd, **kw):
    p = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if p.stdout: print(clean(p.stdout))
    if p.returncode != 0:
        print("----- STDERR (sanitized, complete) -----"); print(clean(p.stderr))
        raise RuntimeError(f"step failed with exit code {p.returncode}: {clean(' '.join(map(str, cmd)))}")
    return p
if MODE == "RUN":  # PREPARE never mounts Drive and never reads a credential
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")  # live approval check; never printed or archived
    SECRETS.append(os.environ["GITHUB_TOKEN"])
BASE = pathlib.Path("/content/ta01-fill-diagnostic-runtime")
BASE.mkdir(exist_ok=False)  # fresh Runtime only
run([sys.executable,"-m","pip","install","uv==0.13.4"])
run([sys.executable,"-m","uv","python","install","3.11.17"])
run([sys.executable,"-m","uv","venv","--python","3.11.17",str(BASE/"venv")])
PYTHON = str(BASE/"venv/bin/python")
def checkout(repository, sha, name):
    target = BASE/name
    run(["git","clone","https://github.com/hh4832/"+repository+".git",str(target)])
    run(["git","-C",str(target),"checkout","--detach",sha])
    assert subprocess.check_output(["git","-C",str(target),"rev-parse","HEAD"],text=True).strip()==sha
    assert not subprocess.check_output(["git","-C",str(target),"status","--porcelain"],text=True).strip()
    return target
CODE = checkout("strategy-research-agent",CODE_SHA,"code")
CORE = checkout("market-research-core",CORE_SHA,"core")
run([sys.executable,"-m","uv","pip","sync","--python",PYTHON,str(CODE/"requirements/phase5c1-runtime.txt")])
run([sys.executable,"-m","uv","pip","install","--python",PYTHON,"--no-deps","--no-build-isolation",
     "-e",str(CORE),"-e",str(CODE)])
settings = dict(mode=MODE, authorization_id=AUTHORIZATION_ID, code_sha=CODE_SHA,
                approval_expires_utc=APPROVAL_EXPIRES_UTC, approval_url=APPROVAL_URL,
                recovery_archive=RECOVERY_ARCHIVE, original_preparation_archive=ORIGINAL_PREPARATION_ARCHIVE,
                confirm_authorization=CONFIRM_AUTHORIZATION, drive_root=DRIVE_ROOT,
                local_staging=str(BASE/"staging"))
SETTINGS = BASE/"settings.json"
SETTINGS.write_text(json.dumps(settings))  # pins/paths only; no credential
run([PYTHON,"-u","-m","strategy_agent.research_only.ta01_fill_diagnostic","--settings",str(SETTINGS)], cwd=CODE)
